<a href="https://colab.research.google.com/github/steveneedham/columbus-micromobility-data/blob/main/Veo_Spin_CBS_GBFS_Extract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import datetime
import pandas as pd
import requests
import csv
import matplotlib.pyplot as plt
import seaborn as sns
import shutil

# ==========================================
# 1. ENVIRONMENT & PATH SETUP
# ==========================================
base_dir = "."
plots_dir = os.path.join(base_dir, "plots")
summary_dir = os.path.join(base_dir, "daily_summary")
snapshots_dir = os.path.join(base_dir, "snapshots")

# Create directories safely using relative paths
for directory in [plots_dir, summary_dir, snapshots_dir]:
    os.makedirs(directory, exist_ok=True)

# Generate one global timestamp to be used across all files in this specific execution
now_utc = datetime.datetime.now(datetime.timezone.utc)
RUN_TIMESTAMP = now_utc.strftime("%Y%m%dT%H%M%SZ")

print(f"✅ Environment setup complete. Run timestamp: {RUN_TIMESTAMP}")

# ==========================================
# 2. DATA EXTRACTION PIPELINE
# ==========================================
def export_fleet_to_csv():
    print("Fetching live vehicle data for Columbus...\n")

    veo_fbs_url = "https://cluster-prod.veoride.com/api/shares/name/cbs/gbfs/free_bike_status"
    veo_vt_url  = "https://cluster-prod.veoride.com/api/shares/name/cbs/gbfs/vehicle_types"
    spin_fbs_url = "https://mds.bird.co/gbfs/v2/public/provider/spin/columbus/free_bike_status.json"
    headers = {'User-Agent': 'Mozilla/5.0 (compatible; fleet-export/1.1)'}

    def meters_to_miles(m):
        try:
            return round(float(m) / 1609.34, 2)
        except Exception:
            return ""

    def is_available(bike: dict) -> bool:
        return bike.get('is_disabled') == 0 and bike.get('is_reserved') == 0

    def get_spin_battery_pct(b: dict):
        candidates = [
            b.get("battery_pct"), b.get("battery_level"), b.get("current_fuel_percent"),
            (b.get("vehicle") or {}).get("battery_pct"), (b.get("vehicle") or {}).get("battery_level"),
            (b.get("attributes") or {}).get("battery_pct"), b.get("charge")
        ]
        for v in candidates:
            if v is None or v == "":
                continue
            try:
                fv = float(v)
                if 0 < fv <= 1:
                    fv *= 100
                fv = max(0, min(100, fv))
                return int(round(fv))
            except Exception:
                return v
        return ""

    def score(rec: dict) -> tuple:
        has_range = 1 if rec.get("Range_Miles") not in ("", None) else 0
        has_type  = 1 if rec.get("Type") not in ("", None) else 0
        last_rep  = rec.get("Last_Reported") or 0
        return (has_range, has_type, last_rep)

    records_by_id = {}
    def upsert(rec: dict):
        vid = rec.get("Vehicle_ID")
        if not vid or vid == "Unknown":
            return
        existing = records_by_id.get(vid)
        if existing is None or score(rec) > score(existing):
            records_by_id[vid] = rec

    # --- Fetch Veo Types ---
    veo_type_map = {}
    try:
        vt_resp = requests.get(veo_vt_url, headers=headers, timeout=15)
        vt_resp.raise_for_status()
        vt_data = vt_resp.json().get("data", {}).get("vehicle_types", [])
        veo_type_map = {v.get("vehicle_type_id"): v.get("form_factor") for v in vt_data}
    except Exception as e:
        print(f"⚠️ Could not fetch Veo vehicle_types: {e}")

    # --- Fetch Veo Data ---
    try:
        veo_resp = requests.get(veo_fbs_url, headers=headers, timeout=15)
        veo_resp.raise_for_status()
        for b in veo_resp.json().get("data", {}).get("bikes", []):
            upsert({
                "Company": "Veo",
                "Vehicle_ID": b.get("bike_id", "Unknown"),
                "Type": veo_type_map.get(b.get("vehicle_type_id"), "") or "",
                "Latitude": b.get("lat"),
                "Longitude": b.get("lon"),
                "Battery_Pct": b.get("battery_level") or b.get("battery_pct") or "",
                "Range_Miles": meters_to_miles(b.get("current_range_meters")),
                "Is_Available": is_available(b),
                "Is_Disabled": b.get("is_disabled"),
                "Is_Reserved": b.get("is_reserved"),
                "Last_Reported": b.get("last_reported") or b.get("last_updated") or 0,
            })
    except Exception as e:
        print(f"❌ Error fetching Veo data: {e}")

    # --- Fetch Spin Data ---
    spin_type_map = {
        "2ea3c8b2-ed07-4c53-b87e-638c08471309": "scooter",
        "bae2102b-56ba-42ba-9097-720e5990b4b2": "e-bike"
    }
    try:
        spin_resp = requests.get(spin_fbs_url, headers=headers, timeout=15)
        spin_resp.raise_for_status()
        for b in spin_resp.json().get("data", {}).get("bikes", []):
            upsert({
                "Company": "Spin",
                "Vehicle_ID": b.get("bike_id") or b.get("vehicle_id") or "Unknown",
                "Type": spin_type_map.get(b.get("vehicle_type_id"), "e-bike"),
                "Latitude": b.get("lat"),
                "Longitude": b.get("lon"),
                "Battery_Pct": get_spin_battery_pct(b),
                "Range_Miles": meters_to_miles(b.get("current_range_meters") or b.get("range_meters")),
                "Is_Available": is_available(b),
                "Is_Disabled": b.get("is_disabled"),
                "Is_Reserved": b.get("is_reserved"),
                "Last_Reported": b.get("last_reported") or b.get("last_updated") or 0,
            })
    except Exception as e:
        print(f"❌ Error fetching Spin data: {e}")

    # --- Write Outputs ---
    rows = list(records_by_id.values())
    if not rows:
        print("No vehicles found.")
        return pd.DataFrame()

    # Save Snapshot CSV
    snapshot_filename = os.path.join(snapshots_dir, f"columbus_scooters_{RUN_TIMESTAMP}.csv")
    df = pd.DataFrame(rows)
    df.to_csv(snapshot_filename, index=False)
    print(f"🎉 Exported {len(rows)} vehicles to '{snapshot_filename}'")

    # Append to Daily Summary
    pd.set_option('future.no_silent_downcasting', True)
    summary = {
        "timestamp": RUN_TIMESTAMP,
        "total_vehicles": len(df),
        "available": df["Is_Available"].sum(),
        "veo_total": len(df[df["Company"]=="Veo"]),
        "spin_total": len(df[df["Company"]=="Spin"]),
        "avg_range": df["Range_Miles"].replace("", 0).astype(float).mean()
    }
    summary_df = pd.DataFrame([summary])
    summary_file = os.path.join(summary_dir, "fleet_summary.csv")

    if os.path.exists(summary_file):
        summary_df.to_csv(summary_file, mode="a", header=False, index=False)
    else:
        summary_df.to_csv(summary_file, index=False)

    print(f"📈 Appended daily summary to '{summary_file}'")

    return df

# Run the extraction
all_vehicles_df = export_fleet_to_csv()

# ==========================================
# 3. DATA VISUALIZATIONS
# ==========================================
if not all_vehicles_df.empty:
    print("Generating visualizations...")

    # Pre-process numeric data for plots
    all_vehicles_df['Battery_Pct_Numeric'] = pd.to_numeric(all_vehicles_df['Battery_Pct'], errors='coerce')
    all_vehicles_df['Range_Miles_Numeric'] = pd.to_numeric(all_vehicles_df['Range_Miles'], errors='coerce')

    # Battery Distribution Plot
    df_battery = all_vehicles_df.dropna(subset=['Battery_Pct_Numeric'])
    if not df_battery.empty:
        plt.figure(figsize=(12, 7))
        sns.histplot(data=df_battery, x='Battery_Pct_Numeric', hue='Company', multiple='stack', kde=True, bins=20)
        plt.title('Vehicle Battery Percentage by Company')
        plt.savefig(os.path.join(plots_dir, 'battery_percentage_distribution.png'))
        plt.close()

    # Range Distribution Plot
    df_range = all_vehicles_df.dropna(subset=['Range_Miles_Numeric'])
    if not df_range.empty:
        plt.figure(figsize=(12, 7))
        sns.histplot(data=df_range, x='Range_Miles_Numeric', hue='Company', multiple='stack', kde=True, bins=30)
        plt.title('Vehicle Range (Miles) by Company')
        plt.savefig(os.path.join(plots_dir, 'range_distribution.png'))
        plt.close()

    # Vehicle Type by Vendor Plot
    type_dist = all_vehicles_df.groupby(['Company', 'Type']).size().reset_index(name='Count')
    plt.figure(figsize=(10, 7))
    sns.barplot(x='Type', y='Count', hue='Company', data=type_dist, palette='viridis')
    plt.title('Vehicle Types by Vendor')
    plt.savefig(os.path.join(plots_dir, 'vehicle_type_distribution.png'))
    plt.close()

    print(f"✅ All visualizations saved to the '{plots_dir}' directory.")

# ==========================================
# 4. CONDITIONAL GOOGLE DRIVE BACKUP
# ==========================================
# This check ensures the script doesn't crash in GitHub Actions
try:
    from google.colab import drive
    in_colab = True
except ImportError:
    in_colab = False

if in_colab:
    print("Detected Google Colab environment. Mounting Drive...")
    drive.mount('/content/drive')

    drive_base_dir = '/content/drive/MyDrive/columbus_micromobility_snapshots'
    current_run_output_dir = os.path.join(drive_base_dir, RUN_TIMESTAMP)
    os.makedirs(current_run_output_dir, exist_ok=True)

    print(f"Backing up data to Google Drive: {current_run_output_dir}")

    subdirs_to_copy = ['snapshots', 'daily_summary', 'plots']
    for subdir in subdirs_to_copy:
        source_path = os.path.join(base_dir, subdir)
        destination_path = os.path.join(current_run_output_dir, subdir)

        if os.path.exists(source_path):
            shutil.copytree(source_path, destination_path, dirs_exist_ok=True)

    print("✅ Backup complete.")
else:
    print("Not running in Google Colab. Skipping Google Drive backup step.")

✅ Environment setup complete. Run timestamp: 20260802T091411Z
Fetching live vehicle data for Columbus...



🎉 Exported 3526 vehicles to './snapshots/columbus_scooters_20260802T091411Z.csv'
📈 Appended daily summary to './daily_summary/fleet_summary.csv'
Generating visualizations...


✅ All visualizations saved to the './plots' directory.
Not running in Google Colab. Skipping Google Drive backup step.
